# SmartRoute Waste · Waste photo classifier

Transfer learning with **MobileNetV3-Large** (ImageNet pre-trained) on public waste-image datasets,
exported to **ONNX** so the SmartRoute server can run it with ONNX Runtime (no GPU needed).

**How to run (about 20–30 min):**
1. `Runtime → Change runtime type → T4 GPU`, then **Save**.
2. Get a Kaggle API token: kaggle.com → profile picture → **Settings** → **API** → **Generate New Token**, and copy it.
3. `Runtime → Run all`. When cell 2 asks, press ENTER and paste the token (or type `j` to upload an old `kaggle.json`).
4. At the end a file **`smartroute_model.zip`** downloads. Unzip it into `backend/app/ai/model/` of the project.

The model learns detailed item classes (battery, cardboard, biological, …) and the server maps them to the
5 SmartRoute waste types: organic, recyclable, hazardous, e_waste, general.

In [ ]:
# 1. Setup
!pip -q install -U kaggle onnx onnxruntime onnxscript
import torch, torchvision, os, json, random, zipfile, glob, time
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '(switch Runtime -> T4 GPU if this says cpu)')

In [ ]:
# 2. Kaggle credentials: paste your API token (new Kaggle) OR upload kaggle.json (old Kaggle)
from google.colab import files
import getpass
os.makedirs('/root/.kaggle', exist_ok=True)
if not (os.path.exists('/root/.kaggle/kaggle.json') or os.environ.get('KAGGLE_API_TOKEN')):
    choice = input('Press ENTER to paste a Kaggle API token, or type j to upload kaggle.json: ').strip().lower()
    if choice == 'j':
        up = files.upload()
        with open('/root/.kaggle/kaggle.json', 'wb') as f:
            f.write(list(up.values())[0])
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    else:
        token = getpass.getpass('Paste your Kaggle API token (it stays hidden): ').strip()
        os.environ['KAGGLE_API_TOKEN'] = token
        with open('/root/.kaggle/access_token', 'w') as f:
            f.write(token)
        os.chmod('/root/.kaggle/access_token', 0o600)
rc = os.system('kaggle datasets list -s garbage --page-size 1 > /dev/null 2>&1')
print('Kaggle login OK' if rc == 0 else '!! Kaggle login failed - check the token (it is shown only once on kaggle.com; generate a new one if needed)')

In [ ]:
# 3. Configuration
MAIN_DATASET = 'mostafaabla/garbage-classification'   # 12 classes, ~15k photos
# Optional e-waste photos: paste any Kaggle e-waste image dataset slug here (search "e-waste image dataset"),
# e.g. 'owner/dataset-name'. Leave '' to skip. Every folder in it becomes an e_waste item class
# (a 'battery' folder is still treated as hazardous).
EWASTE_DATASET = ''
UPLOAD_EXTRA_ZIP = False      # True -> upload a zip whose sub-folders are class names (e.g. e_waste/, organic/)
MAX_PER_CLASS = 1200          # cap big classes so the model is not biased towards them
EPOCHS = 10
IMG = 224
SEED = 42

# detailed item class -> SmartRoute waste type
TYPE_OF = {
    'biological': 'organic', 'food': 'organic', 'organic': 'organic', 'vegetable': 'organic', 'garden': 'organic',
    'paper': 'recyclable', 'cardboard': 'recyclable', 'plastic': 'recyclable', 'metal': 'recyclable',
    'glass': 'recyclable', 'brown-glass': 'recyclable', 'green-glass': 'recyclable', 'white-glass': 'recyclable',
    'battery': 'hazardous', 'batteries': 'hazardous', 'hazardous': 'hazardous',
    'clothes': 'general', 'shoes': 'general', 'trash': 'general', 'general': 'general',
    'e_waste': 'e_waste', 'e-waste': 'e_waste', 'ewaste': 'e_waste',
}
random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 4. Download datasets and collect (image path, item class)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

def download(slug, dest):
    if not slug:
        return None
    os.makedirs(dest, exist_ok=True)
    rc = os.system(f'kaggle datasets download -d {slug} -p {dest} --unzip -q')
    if rc != 0:
        print(f'!! Could not download {slug}. Check the slug on kaggle.com (owner/dataset-name).')
        return None
    return dest

def collect(root, force_type=None):
    items = []
    for d, _, fs in os.walk(root):
        imgs = [f for f in fs if f.lower().endswith(IMG_EXT)]
        if not imgs:
            continue
        name = os.path.basename(d).strip().lower().replace(' ', '-').replace('_', '-')
        name = 'e_waste' if name in ('e-waste', 'ewaste') else name
        if force_type == 'e_waste' and name not in ('battery', 'batteries'):
            label = f'ewaste-{name}'
            TYPE_OF[label] = 'e_waste'
        else:
            label = name
        if label not in TYPE_OF:
            print('  skipping folder (unknown class):', d)
            continue
        items += [(os.path.join(d, f), label) for f in imgs]
    return items

samples = []
root = download(MAIN_DATASET, '/content/data/main')
if root:
    samples += collect(root)
eroot = download(EWASTE_DATASET, '/content/data/ewaste')
if eroot:
    samples += collect(eroot, force_type='e_waste')
if UPLOAD_EXTRA_ZIP:
    up = files.upload()
    for name, data in up.items():
        open('/content/extra.zip', 'wb').write(data)
        zipfile.ZipFile('/content/extra.zip').extractall('/content/data/extra')
    samples += collect('/content/data/extra')

# de-duplicate and cap per class
by_cls = {}
for p, c in dict(samples).items():
    by_cls.setdefault(c, []).append(p)
for c in by_cls:
    random.shuffle(by_cls[c])
    by_cls[c] = by_cls[c][:MAX_PER_CLASS]
CLASSES = sorted(by_cls)
print(f'{sum(len(v) for v in by_cls.values())} images, {len(CLASSES)} item classes:')
for c in CLASSES:
    print(f'  {c:28s} {len(by_cls[c]):5d}  -> {TYPE_OF[c]}')
assert len(CLASSES) >= 2, 'No images found - check the dataset download above.'

In [ ]:
# 5. Train / validation / test split (70 / 15 / 15, stratified)
from sklearn.model_selection import train_test_split
X = [p for c in CLASSES for p in by_cls[c]]
Y = [CLASSES.index(c) for c in CLASSES for _ in by_cls[c]]
Xtr, Xtmp, Ytr, Ytmp = train_test_split(X, Y, test_size=0.30, stratify=Y, random_state=SEED)
Xva, Xte, Yva, Yte = train_test_split(Xtmp, Ytmp, test_size=0.50, stratify=Ytmp, random_state=SEED)
print('train', len(Xtr), '| val', len(Xva), '| test', len(Xte))

In [ ]:
# 6. Data pipeline
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([T.RandomResizedCrop(IMG, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.RandomRotation(15),
                      T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(IMG), T.ToTensor(), T.Normalize(MEAN, STD)])

class Imgs(Dataset):
    def __init__(self, xs, ys, tf): self.xs, self.ys, self.tf = xs, ys, tf
    def __len__(self): return len(self.xs)
    def __getitem__(self, i):
        return self.tf(Image.open(self.xs[i]).convert('RGB')), self.ys[i]

dl_tr = DataLoader(Imgs(Xtr, Ytr, train_tf), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
dl_va = DataLoader(Imgs(Xva, Yva, eval_tf), batch_size=128, num_workers=2)
dl_te = DataLoader(Imgs(Xte, Yte, eval_tf), batch_size=128, num_workers=2)
counts = torch.bincount(torch.tensor(Ytr), minlength=len(CLASSES)).float()
class_w = (counts.sum() / (len(CLASSES) * counts)).to(DEVICE)   # balance rare classes

In [ ]:
# 7. Model: MobileNetV3-Large, ImageNet weights, new classification head
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V2)
model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(CLASSES))
model = model.to(DEVICE)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.1)

def evaluate(dl):
    model.eval(); correct = n = 0; preds = []; ys = []
    with torch.no_grad():
        for x, y in dl:
            p = model(x.to(DEVICE)).argmax(1).cpu()
            preds += p.tolist(); ys += y.tolist()
    return sum(int(a == b) for a, b in zip(preds, ys)) / len(ys), preds, ys

# phase 1: train only the new head (2 epochs), phase 2: fine-tune everything
history, best, best_state = [], 0.0, None
for epoch in range(EPOCHS):
    fine = epoch >= 2
    for p in model.features.parameters():
        p.requires_grad = fine
    if epoch in (0, 2):
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.AdamW(params, lr=1e-3 if not fine else 3e-4, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=(2 if not fine else EPOCHS - 2) * len(dl_tr))
    model.train(); t0 = time.time(); tot = 0.0
    for x, y in dl_tr:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step(); sched.step()
        tot += loss.item() * len(y)
    va, _, _ = evaluate(dl_va)
    history.append({'epoch': epoch + 1, 'loss': round(tot / len(Xtr), 4), 'val_acc': round(va, 4)})
    print(f'epoch {epoch + 1}/{EPOCHS}  loss {tot / len(Xtr):.3f}  val acc {va:.3f}  ({time.time() - t0:.0f}s)')
    if va > best:
        best, best_state = va, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print('best validation accuracy', round(best, 4))

In [ ]:
# 8. Test-set evaluation (item classes and the 5 SmartRoute waste types) + confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
acc, preds, ys = evaluate(dl_te)
TYPES = ['organic', 'recyclable', 'general', 'e_waste', 'hazardous']
types_present = [t for t in TYPES if any(TYPE_OF[c] == t for c in CLASSES)]
tp = [TYPE_OF[CLASSES[i]] for i in preds]
ty = [TYPE_OF[CLASSES[i]] for i in ys]
type_acc = sum(a == b for a, b in zip(tp, ty)) / len(ty)
print(f'TEST accuracy - item class: {acc:.3f} | SmartRoute waste type: {type_acc:.3f}')
report = classification_report(ty, tp, labels=types_present, output_dict=True, zero_division=0)
print(classification_report(ty, tp, labels=types_present, zero_division=0))
cm = confusion_matrix(ty, tp, labels=types_present)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=types_present).plot(ax=ax, cmap='Greens', colorbar=False)
ax.set_title(f'SmartRoute waste type - test accuracy {type_acc:.1%}')
plt.tight_layout(); plt.savefig('/content/confusion_matrix.png', dpi=160); plt.show()

In [ ]:
# 9. Export to ONNX, check it matches PyTorch, and package for the server
import onnxruntime as ort, numpy as np
model.eval().cpu()
dummy = torch.randn(1, 3, IMG, IMG)
torch.onnx.export(model, dummy, '/content/waste_classifier.onnx', input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}}, opset_version=17, dynamo=False)
sess = ort.InferenceSession('/content/waste_classifier.onnx', providers=['CPUExecutionProvider'])
diff = np.abs(sess.run(None, {'input': dummy.numpy()})[0] - model(dummy).detach().numpy()).max()
print('max difference PyTorch vs ONNX:', float(diff))
meta = {
    'labels': CLASSES, 'type_of': {c: TYPE_OF[c] for c in CLASSES}, 'size': IMG, 'resize': 256,
    'mean': MEAN, 'std': STD, 'architecture': 'MobileNetV3-Large (ImageNet) fine-tuned',
    'datasets': [d for d in (MAIN_DATASET, EWASTE_DATASET) if d],
    'images': {'train': len(Xtr), 'val': len(Xva), 'test': len(Xte)},
    'test_accuracy_item': round(acc, 4), 'test_accuracy_type': round(type_acc, 4),
    'per_type': {t: {k: round(v, 3) for k, v in report[t].items()} for t in types_present},
    'confusion_matrix': {'labels': types_present, 'matrix': cm.tolist()},
    'history': history, 'trained': time.strftime('%Y-%m-%d'),
}
json.dump(meta, open('/content/labels.json', 'w'), indent=1)
with zipfile.ZipFile('/content/smartroute_model.zip', 'w') as z:
    for f in ('waste_classifier.onnx', 'labels.json', 'confusion_matrix.png'):
        z.write(f'/content/{f}', f)
print('size of model:', round(os.path.getsize('/content/waste_classifier.onnx') / 1e6, 1), 'MB')
files.download('/content/smartroute_model.zip')

**Done.** Unzip `smartroute_model.zip` into `backend/app/ai/model/` so you have:

```
backend/app/ai/model/waste_classifier.onnx
backend/app/ai/model/labels.json
backend/app/ai/model/confusion_matrix.png
```

Then commit and push. The server picks the model up automatically; the dashboard shows its accuracy on **AI status**.